# INSTALL

In [1]:
!pip install typhoon-ocr pdf2image Pillow

In [2]:
!apt-get install -y poppler-utils

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!pip install -U google-generativeai

In [4]:
!pip install python-dotenv

In [ ]:
# !pip install -U transformers accelerate torch

In [ ]:
# !pip install -U transformers accelerate

In [6]:

# !pip uninstall -y transformers accelerate

!pip install transformers==4.3.0 accelerate 

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 8.7 MB/s  0:00:00
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ---------------------------------------- 0.0/897.5 kB ? eta -:--:--
   ---------------------------------------- 897.5/897.5 kB 12.7 MB/s  0:00:00
Failed to build tokenizers


  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [62 lines of output]
      C:\Users\sukon\AppData\Local\Temp\pip-build-env-jioc_vb9\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

# IMPORT

In [ ]:
import os
import json
import re
import requests
# import dashscope
import shutil
from pdf2image import convert_from_path
from typhoon_ocr import ocr_document
import google.generativeai as genai
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

# Pipeline

In [ ]:
class Extractor:
    # รายชื่อพรรคการเมืองที่ถูกต้องสำหรับใช้ตรวจสอบ (Validation)
    VALID_PARTIES = """
    ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เครือข่ายชาวนาแห่งประเทศไทย, เพื่อไทย, 
    ชาติพัฒนา, ชาติไทยพัฒนา, อนาคตไทย, ภูมิใจไทย, สังคมประชาธิปไตยไทย, รักชาติ, 
    ประชาธิปไตยใหม่, พลังบูรพา, ครูไทยเพื่อประชาชน, พลังท้องถิ่นไท, ประชาชน, 
    ไทยก้าวใหม่, เสรีรวมไทย, รักษ์ธรรม, พลังประชาธิปไตย, พลังสุราษฎร์, พลังไทยรักชาติ, 
    เพื่อชีวิตใหม่, ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม, 
    รวมพลัง, ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, พร้อมพัฒนา, ประชาชาติ, แผ่นดินธรรม, 
    คลองไทย, พลังประชารัฐ, เศรษฐกิจใหม่, พลังสังคม, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, 
    กรีน, วิชชั่นใหม่, พลวัต, กล้าธรรม, ไทยรวมไทย, กล้า, ฟิวชัน, พลังสังคมใหม่, 
    ไทยสร้างไทย, รวมไทยสร้างชาติ, มิติใหม่, ไทยสมาร์ท, ไทยภักดี, ไทยพิทักษ์ธรรม, 
    ไทยชนะ, ไทรวมพลัง, ราษฎร์วิถี, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, 
    ไทยก้าวหน้า, ตะวันใหม่, พร้อม, รวมใจไทย, สัมมาธิปไตย, รักภูเก็ต, ประชาอาสาชาติ, 
    ไทยทรัพย์ทวี, รวมพลังประชาชน, อนาคตไกล, ยางพาราไทย, เพื่อบ้านเมือง
    """

    def __init__(self, typhoon_key: str, gemini_key: str = None, qwen_model_name: str = "Qwen/Qwen2.5-7B-Instruct"):
        """Initializes the OCR engine and the LLMs (Qwen and Gemini)."""
        # Setup Typhoon OCR
        os.environ["TYPHOON_OCR_API_KEY"] = typhoon_key
        
        # Setup Gemini (Optional)
        if gemini_key:
            genai.configure(api_key=gemini_key)
            self.gemini_model = genai.GenerativeModel('gemini-2.5-flash')
        else:
            self.gemini_model = None
        
        # Setup Qwen Local
        print(f"Loading Qwen model: {qwen_model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(qwen_model_name, trust_remote_code=True)
        
        # เพิ่ม trust_remote_code=True เพื่อป้องกันปัญหาการโหลดไฟล์ configuration บางตัวของ Qwen
        self.qwen_model = AutoModelForCausalLM.from_pretrained(
            qwen_model_name,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True
        )

    def extract_pdf(self, pdf_path: str, use_model: str = "qwen") -> str:
        """Process PDF: Multi-page OCR -> Merged Text -> LLM Parsing."""
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"The file {pdf_path} was not found.")
        
        print(f"Starting Extraction: {pdf_path}")
        
        temp_dir = "temp_pages"
        if not os.path.exists(temp_dir):
            os.makedirs(temp_dir)
            
        full_markdown_text = ""
        
        try:
            # แปลง PDF เป็นภาพทีละหน้าเพื่อส่งให้ OCR
            pages = convert_from_path(pdf_path)
            
            for i, page in enumerate(pages):
                page_path = os.path.join(temp_dir, f"page_{i+1}.jpg")
                page.save(page_path, "JPEG")
                
                print(f"--- OCRing Page {i+1}/{len(pages)} ---")
                page_markdown = ocr_document(page_path)
                full_markdown_text += f"\n--- PAGE {i+1} ---\n" + page_markdown
            
            # เลือกว่าจะใช้ Model ตัวไหนในการ Parse ข้อมูล
            if use_model.lower() == "gemini" and self.gemini_model:
                print("Parsing with Gemini...")
                return self._parse_markdown_byGemini(full_markdown_text)
            else:
                print("Parsing with Qwen (Local)...")
                return self._parse_markdown_byQwen(full_markdown_text)
            
        finally:
            # ลบไฟล์ภาพชั่วคราว
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)

    def _get_shared_prompt(self, full_text: str) -> str:
        """Shared prompt logic for both models to ensure consistency."""
        return f"""
        Extract the election results from the following Thai OCR text into a structured JSON format.
        
        Requirements:
        1. Fix any OCR typos in province, district, or party names.
        2. Convert all numbers (including Thai digits) to standard integers.
        3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
           and a 'results' list (party number, name, and votes).
        4. **PARTY NAME VALIDATION**: Compare the party name from OCR with the following valid list. 
           If the OCR name is misspelled, change it to the correct name from this list:
           {self.VALID_PARTIES}
        5. Check the score: look for numbers and Thai text in parentheses (). 
           If the main digit is unreadable, convert the Thai text description into an integer.

        OCR Text:
        {full_text}
        """

    def _parse_markdown_byGemini(self, full_text: str) -> str:
        prompt = self._get_shared_prompt(full_text)
        response = self.gemini_model.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"}
        )
        return response.text

    def _parse_markdown_byQwen(self, full_text: str) -> str:
        system_prompt = "You are Qwen, a helpful assistant created by Alibaba Cloud. You are an expert in Thai language and data extraction."
        user_prompt = self._get_shared_prompt(full_text)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.qwen_model.device)

        # Generate using local model
        generated_ids = self.qwen_model.generate(
            **model_inputs,
            max_new_tokens=2048,
            temperature=0.1
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return self._clean_json_string(response)

    def _clean_json_string(self, content: str) -> str:
        """Removes Markdown code blocks from the string."""
        content = content.strip()
        if content.startswith("```json"):
            content = content[7:]
        elif content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
        return content.strip()

In [ ]:
if __name__ == "__main__":

    load_dotenv()
    
    TYPHOON_KEY = os.getenv("TYPHOON_KEY")
    GEMINI_KEY = os.getenv("GEMINI_KEY")
    
    extractor = Extractor(TYPHOON_KEY, GEMINI_KEY)
    
    try:
        json_output = extractor.extract_pdf("doc1.pdf")
        print("--- Extraction Result ---")
        print(json_output)
        
        # Save locally
        with open("output_gemini.json", "w", encoding="utf-8") as f:
            f.write(json_output)
            
    except Exception as e:
        print(f"An error occurred: {e}")